In [1]:
import heapq
import math

In [2]:
class Node:
    """
    Represents a discrete search state in grid space.
    Priority in the min-heap is governed by total evaluation cost f = g + h.
    Tie-breaking prioritizes smaller heuristic distance h to drive exploration toward the goal.
    """
    def __init__(self, position, parent=None, g=0.0, h=0.0):
        self.position = position  # Coordinate tuple: (row, col)
        self.parent = parent      # Reference to predecessor Node instance
        self.g = g                # Cumulative path cost from start to current node
        self.h = h                # Heuristic estimate from current node to goal
        self.f = g + h            # Evaluation function priority

    def __lt__(self, other):
        if self.f == other.f:
            return self.h < other.h
        return self.f < other.f

    def __eq__(self, other):
        return self.position == other.position

    def __hash__(self):
        return hash(self.position)

In [3]:
def manhattan_distance(p1, p2):
    """Calculates L1 norm distance between two 2D grid coordinates."""
    return abs(p1[0] - p2[0]) + abs(p1[1] - p2[1])

In [4]:
def solve_karp_city(grid, start, goal, portals, portal_cost=1.0):
    """
    Finds the optimal path for a Karp City hovercraft utilizing warp portals.

    Parameters:
        grid (list of list of int): 0 = Open Road, 1 = Building.
        start (tuple): Start coordinate (row, col).
        goal (tuple): Goal coordinate (row, col).
        portals (dict): Mapping (entry_row, entry_col) -> (exit_row, exit_col).
        portal_cost (float): Traversal cost for using a portal jump (default 1.0).

    Returns:
        tuple: (optimal_path_list, minimum_total_cost)
    """
    rows = len(grid)
    cols = len(grid[0])

    start_node = Node(start, None, g=0.0, h=manhattan_distance(start, goal))
    open_heap = []
    heapq.heappush(open_heap, start_node)

    open_dict = {start: start_node}
    closed_set = set()

    cardinal_directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]

    while open_heap:
        current_node = heapq.heappop(open_heap)

        if current_node.position in closed_set:
            continue

        closed_set.add(current_node.position)

        if current_node.position == goal:
            path = []
            curr = current_node
            while curr:
                path.append(curr.position)
                curr = curr.parent
            return path[::-1], current_node.g

        r, c = current_node.position

        # Candidate moves list stores tuples of: (neighbor_position, move_step_cost)
        candidate_moves = []

        # -------------------------------------------------------------
        # TODO 1: Populate candidate_moves with standard 4-directional
        #         cardinal moves (cost = 1.0) if unblocked (grid cell == 0).
        # -------------------------------------------------------------
        # <INSERT YOUR CODE HERE>
        for dr, dc in cardinal_directions:
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] == 0:
                candidate_moves.append(((nr, nc), 1.0))

        # -------------------------------------------------------------
        # TODO 2: Check if current position (r, c) is a portal entry.
        #         If yes, append (portal_exit, portal_cost) to candidate_moves.
        # -------------------------------------------------------------
        # <INSERT YOUR CODE HERE>
        if (r, c) in portals:
            candidate_moves.append((portals[(r, c)], portal_cost))

        for neighbor_pos, step_cost in candidate_moves:
            if neighbor_pos in closed_set:
                continue

            tentative_g = current_node.g + step_cost

            if neighbor_pos in open_dict and tentative_g >= open_dict[neighbor_pos].g:
                continue

            neighbor_node = Node(
                position=neighbor_pos,
                parent=current_node,
                g=tentative_g,
                h=manhattan_distance(neighbor_pos, goal)
            )

            open_dict[neighbor_pos] = neighbor_node
            heapq.heappush(open_heap, neighbor_node)

    return None, float('inf')

In [7]:
# 0 = Open Road, 1 = Impassable Building
karp_grid = [
    [0, 0, 1, 0, 0],
    [1, 0, 1, 0, 1],
    [0, 0, 1, 0, 0],
    [0, 1, 1, 1, 0],
    [0, 0, 0, 0, 0]
]

# Portal mapping dictionary
karp_portals = {
    (0, 1): (0, 3),  # Bypasses building wall at col 2
    (2, 0): (4, 4)   # Teleports directly to destination area
}

start_pos = (0, 0)
goal_pos = (4, 4)

path, cost = solve_karp_city(karp_grid, start_pos, goal_pos, karp_portals, portal_cost=1.0)
print("Hovercraft Path:", path)
print("Total Traversal Cost:", cost)

Hovercraft Path: [(0, 0), (0, 1), (0, 3), (1, 3), (2, 3), (2, 4), (3, 4), (4, 4)]
Total Traversal Cost: 7.0
